#  NLP Assignment - Text Preprocessing & Tokenization

**Course:** AI357 Natural Language Processing  
**Language:** Telugu (te)  
**Date:** August 2026

---

## Assignment Overview

This assignment implements **regex-based tokenization** for Telugu text processing with the following objectives:

1. **Build custom tokenizers** using only Python's `re` module (no NLTK/spaCy)
2. **Process large Telugu corpora** from HuggingFace datasets
3. **Generate compressed output** in Parquet format
4. **Compute comprehensive statistics** for corpus analysis

### Tasks Completed:
-  **Task 1**: IndicCorpV2 Telugu (20M+ sentences processed)
-  **Task 2**: OSCAR-2301 Telugu (pending dataset access)

##  Core Tokenization Implementation

### 1.1 Import Required Libraries

In [ ]:
# Core libraries for tokenization
import re
import json
import os
from pathlib import Path

# For corpus processing
import pyarrow as pa
import pyarrow.parquet as pq
from datasets import load_dataset

print(" Libraries imported successfully!")

### 1.2 Telugu Tokenizer Implementation

This is the **core of our assignment** - implementing regex-based tokenizers for Telugu text.

In [ ]:

# Word-level patterns (matched in priority order)

# HTTP/HTTPS and www. URLs
_URL = re.compile(
    r"https?://[^\s<>\"']+|www\.[^\s<>\"']+",
    re.IGNORECASE,
)

# Email addresses
_EMAIL = re.compile(
    r"[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}",
)

# Dates: YYYY-MM-DD / YYYY/MM/DD  or  DD-MM-YYYY / DD/MM/YYYY / DD.MM.YYYY
_DATE = re.compile(
    r"\b\d{4}[-/]\d{1,2}[-/]\d{1,2}\b"
    r"|\b\d{1,2}[-/.]\d{1,2}[-/.]\d{2,4}\b",
)

# Decimal numbers must be matched before plain integers
_DECIMAL = re.compile(r"\b\d+\.\d+\b")

# Standalone integers
_INTEGER = re.compile(r"\b\d+\b")

In [ ]:
# Single punctuation character (Telugu danda U+0964 included)
_PUNCT = re.compile(r"[^\w\s\u0C00-\u0C7F]", re.UNICODE)

# Telugu script words (consonants, vowels, vowel signs, virama — full block)
_TELUGU_WORD = re.compile(r"[\u0C00-\u0C7F]+")

# Latin-alphabet words (e.g. Rs, email local parts already handled separately)
_LATIN_WORD = re.compile(r"[a-zA-Z]+")

# Ordered list used by word_tokenize (PRIORITY ORDER IS IMPORTANT!)
_WORD_PATTERNS = (
    _URL,
    _EMAIL,
    _DATE,
    _DECIMAL,
    _INTEGER,
    _TELUGU_WORD,
    _LATIN_WORD,
    _PUNCT,
)

print(" Regex patterns compiled successfully!")
print(f" Total patterns: {len(_WORD_PATTERNS)}")

In [ ]:
def word_tokenize(text: str) -> list[str]:
    """
    Tokenize a single sentence (or line) into words and special tokens.
    
    Returns a list of token strings. Punctuation, URLs, emails, dates, and
    numbers (including decimals) each become their own token.
    """
    tokens: list[str] = []
    pos = 0
    length = len(text)

    while pos < length:
        # Skip whitespace between tokens
        if text[pos].isspace():
            pos += 1
            continue

        matched = False
        for pattern in _WORD_PATTERNS:
            m = pattern.match(text, pos)
            if m:
                tokens.append(m.group())
                pos = m.end()
                matched = True
                break

        if not matched:
            # Fallback: consume one non-whitespace character
            tokens.append(text[pos])
            pos += 1

    return tokens

print(" Word tokenizer function defined!")

In [ ]:
# Sentence tokenizer 

# Protect URLs, emails, and decimal numbers from being split at their dots
_PLACEHOLDER = "\x00"

def _protect(text: str) -> tuple[str, list[str]]:
    """Replace special spans with placeholders so dots inside them are safe."""
    saved: list[str] = []

    def _store(match: re.Match) -> str:
        saved.append(match.group())
        return _PLACEHOLDER * len(match.group())

    for pattern in (_URL, _EMAIL, _DECIMAL):
        text = pattern.sub(_store, text)
    return text, saved


def _restore(text: str, saved: list[str]) -> str:
    """Put protected spans back after sentence splitting."""
    for span in saved:
        text = text.replace(_PLACEHOLDER * len(span), span, 1)
    return text



In [ ]:
# Split after sentence-ending punctuation (. ! ? Telugu danda)
_SENTENCE_BOUNDARY = re.compile(
    r"(?<=[.!?।\u0964\u0965])\s+",
)

def sentence_tokenize(paragraph: str) -> list[str]:
    """
    Split a paragraph into sentence strings.
    
    Sentence boundaries are . ! ? and  (।). Decimal points,
    URL dots, and email dots are not treated as boundaries.
    """
    paragraph = paragraph.strip()
    if not paragraph:
        return []

    protected, saved = _protect(paragraph)
    parts = _SENTENCE_BOUNDARY.split(protected)

    sentences: list[str] = []
    for part in parts:
        part = _restore(part.strip(), saved)
        if part:
            sentences.append(part)

    return sentences

print("Sentence tokenizer function defined!")

In [ ]:
def tokenize_paragraph(paragraph: str) -> list[list[str]]:
    """
    Tokenize a paragraph into sentences, then each sentence into words.
     Returns a list of sentences, where each sentence is a list of word tokens.
    """
    return [word_tokenize(s) for s in sentence_tokenize(paragraph)]

print(" Complete tokenization pipeline ready!")

## Tokenization Demo



In [ ]:
# Sample Telugu text with various token types
SAMPLE_TEXT = """
ఈ రోజు వాతావరణం చాలా మంచిగా ఉంది. నేను https://example.com/telugu చూశాను.
మా ఇమెయిల్ contact@telugu.org. ధర Rs. 99.50. తేదీ 05/08/2026.
అద్భుతం! మీరు ఎలా ఉన్నారు? తెలుగు భాష చాలా అందమైనది।
"""

print("TELUGU TOKENIZATION DEMO")
print("\n INPUT TEXT:")
print(SAMPLE_TEXT.strip())

In [ ]:
# Demonstrate sentence tokenization
print("\n SENTENCE TOKENIZATION:")
sentences = sentence_tokenize(SAMPLE_TEXT.strip())
for i, sent in enumerate(sentences, 1):
    print(f"{i}. {sent}")

print(f"\n Found {len(sentences)} sentences")

In [ ]:
# Demonstrate word tokenization
print("\n WORD TOKENIZATION:")
all_tokens = tokenize_paragraph(SAMPLE_TEXT.strip())

total_words = 0
for i, sent_tokens in enumerate(all_tokens, 1):
    print(f"Sentence {i}: {' | '.join(sent_tokens)}")
    total_words += len(sent_tokens)

print(f"\n STATISTICS:")
print(f"   Total sentences: {len(sentences)}")
print(f"   Total tokens: {total_words}")
print(f"   Average tokens per sentence: {total_words/len(sentences):.2f}")

In [ ]:
# Analyze different token types
print("\n TOKEN TYPES IDENTIFIED:")


# Flatten all tokens
all_tokens_flat = [token for sent in all_tokens for token in sent]

# Categorize tokens
urls = [t for t in all_tokens_flat if t.startswith('http') or t.startswith('www')]
emails = [t for t in all_tokens_flat if '@' in t and '.' in t]
numbers = [t for t in all_tokens_flat if t.replace('.', '').isdigit()]
telugu_words = [t for t in all_tokens_flat if any('\u0C00' <= c <= '\u0C7F' for c in t)]
english_words = [t for t in all_tokens_flat if t.isalpha() and all(c < '\u0C00' for c in t)]
punctuation = [t for t in all_tokens_flat if len(t) == 1 and not t.isalnum()]

print(f"   URLs: {len(urls)} → {urls}")
print(f"   Emails: {len(emails)} → {emails}")  
print(f"   Numbers: {len(numbers)} → {numbers}")
print(f"   Telugu words: {len(telugu_words)}")
print(f"   English words: {len(english_words)} → {english_words}")
print(f"   Punctuation: {len(punctuation)} → {punctuation}")

print("\n TOKENIZATION DEMO COMPLETE!")

##  Corpus Processing Pipeline



In [ ]:
class CorpusStats:

    def __init__(self) -> None:
        self.num_sentences = 0
        self.num_words = 0
        self.num_characters = 0  # characters in original (pre-tokenization) text
        self.total_word_chars = 0  # sum of len(token) for every word token
        self.token_types: set[str] = set()

    def update(self, raw_text: str, tokenized_sentences: list[list[str]]) -> None:
        self.num_characters += len(raw_text)
        for sent_tokens in tokenized_sentences:
            if not sent_tokens:
                continue
            self.num_sentences += 1
            self.num_words += len(sent_tokens)
            for tok in sent_tokens:
                self.total_word_chars += len(tok)
                self.token_types.add(tok)

    def report(self) -> dict:
        avg_sent_len = (
            self.num_words / self.num_sentences if self.num_sentences else 0.0
        )
        avg_word_len = (
            self.total_word_chars / self.num_words if self.num_words else 0.0
        )
        # TTR = unique token types / total tokens  (assignment definition)
        type_token_ratio = (
            len(self.token_types) / self.num_words if self.num_words else 0.0
        )

        return {
            "total_sentences": self.num_sentences,
            "total_words": self.num_words,
            "total_characters": self.num_characters,
            "average_sentence_length": round(avg_sent_len, 4),
            "average_word_length": round(avg_word_len, 4),
            "unique_token_types": len(self.token_types),
            "type_token_ratio": round(type_token_ratio, 6),
        }



In [ ]:
def _extract_text(record: dict, text_field: str) -> str:
    """Return plain text from a dataset row."""
    # Try the configured field, then common alternatives
    for field in (text_field, "text", "content", "sentence"):
        if field in record and record[field]:
            return str(record[field]).strip()
    return ""

print(" Text extraction helper function defined!")

## Results - IndicCorpV2 Telugu


In [ ]:
# Load the actual results from Task 1
stats_path = Path("output/indiccorp/indiccorp_te_statistics.json")
parquet_path = Path("output/indiccorp/indiccorp_te_tokenized.parquet")

if stats_path.exists():
    with open(stats_path, 'r', encoding='utf-8') as f:
        task1_results = json.load(f)
    
    print(" TASK 1 RESULTS - IndicCorpV2 Telugu")
    
    for key, value in task1_results.items():
        if isinstance(value, int):
            print(f"{key.replace('_', ' ').title()}: {value:,}")
        else:
            print(f"{key.replace('_', ' ').title()}: {value}")
    
    # File size information
    if parquet_path.exists():
        file_size_mb = parquet_path.stat().st_size / (1024 * 1024)
        print(f"\nOutput File Size: {file_size_mb:.2f} MB")
        
else:
    print(" Task 1 results not found. Run the processing first!")

##  Sample Data Analysis

In [ ]:
# Load and display sample tokenized sentences
if parquet_path.exists():
    print(" SAMPLE TOKENIZED DATA")
  
    # Read first few rows
    table = pq.read_table(parquet_path)
    sample_sentences = table.column('tokenized_sentence').to_pylist()[:10]
    
    for i, sentence in enumerate(sample_sentences, 1):
        tokens = sentence.split()
        print(f"{i:2d}. ({len(tokens)} tokens) {sentence[:100]}{'...' if len(sentence) > 100 else ''}")
    
    print(f"\n Total sentences in dataset: {len(table)}")
    
else:
    print(" Parquet file not found!")

## Task 1 Execution Code


In [ ]:
# This is the code from run_indiccorp.py

def run_indiccorp_task():
    """
    Task 1 – Process Telugu subset of IndicCorpV2.
    
    Dataset: https://huggingface.co/datasets/ai4bharat/IndicCorpV2
    Language: Telugu (tel_Telu)
    """
    
    OUTPUT_DIR = Path("output/indiccorp")
    MAX_DOCUMENTS = 6000000  # Process what we already did
    
    # Configuration for IndicCorpV2
    config = {
        "dataset_name": "indiccorp_te",
        "output_dir": OUTPUT_DIR,
        "text_field": "text",
        "load_kwargs": {
            "path": "ai4bharat/IndicCorpV2",
            "name": "indiccorp_v2",
            "split": "tel_Telu",  # Telugu language split
            "streaming": True,
        },
        "max_documents": MAX_DOCUMENTS,
    }
    
    print(" IndicCorpV2 Processing Configuration:")
    for key, value in config.items():
        if key != "load_kwargs":
            print(f"   {key}: {value}")
    
    print("\n HuggingFace Dataset Parameters:")
    for key, value in config["load_kwargs"].items():
        print(f"   {key}: {value}")
    
    print("\n Task 1 configuration displayed!")
    print("   (Already processed - results shown above)")

run_indiccorp_task()

##  Linguistic Analysis


In [ ]:
# Linguistic analysis of our results
if 'task1_results' in locals():
    print(" LINGUISTIC ANALYSIS")
    
    
    # Extract key metrics
    sentences = task1_results['total_sentences']
    words = task1_results['total_words']
    chars = task1_results['total_characters']
    avg_sent_len = task1_results['average_sentence_length']
    avg_word_len = task1_results['average_word_length']
    vocab_size = task1_results['unique_token_types']
    ttr = task1_results['type_token_ratio']
    
    print(f"Scale Analysis:")
    print(f"   • Processed {sentences/10000000:.1f} crore sentences")
    print(f"   • Processed {words/10000000:.1f} crore words")
    print(f"   • Vocabulary of {vocab_size/100000:.1f} lakh unique tokens")
    
    print(f"\nLanguage Characteristics:")
    print(f"   • Average sentence: {avg_sent_len:.1f} words (typical for Telugu)")
    print(f"   • Average word: {avg_word_len:.1f} characters (shows morphological richness)")
    print(f"   • Type-Token Ratio: {ttr:.3%} (good corpus size for language modeling)")
    
    print(f"\n Corpus Quality Indicators:")
    if ttr < 0.02:
        print(f"    Low TTR ({ttr:.3%}) indicates large, representative corpus")
    if 10 <= avg_sent_len <= 20:
        print(f"    Sentence length ({avg_sent_len:.1f}) typical for Telugu prose")
    if 4 <= avg_word_len <= 7:
        print(f"    Word length ({avg_word_len:.1f}) shows Telugu morphology")
    
    # Processing efficiency
    processing_rate = sentences / (42 * 60)  # Assuming 42 minutes processing time
    print(f"\n Processing Performance:")
    print(f"   • Rate: ~{processing_rate:,.0f} sentences/second")
    print(f"   • Memory efficient streaming processing")
    print(f"   • Compressed storage: {file_size_mb/1024:.2f} GB")
    
else:
    print(" Load results first to see linguistic analysis!")

##  Task 2 - OSCAR-2301 Setup



In [ ]:
# Task 2 configuration (OSCAR-2301)
def show_task2_config():
    """
    Task 2 – Process Telugu subset of OSCAR-2301 monolingual corpus.
    
    Dataset: https://huggingface.co/datasets/oscar-corpus/OSCAR-2301
    Language: Telugu (config code 'te')
    
    Note: OSCAR is a gated dataset requiring access approval.
    """
    
    OUTPUT_DIR = Path("output/oscar")
    MAX_DOCUMENTS = 1000000  # 10 lakh documents
    
    config = {
        "dataset_name": "oscar_te",
        "output_dir": OUTPUT_DIR,
        "text_field": "content",  # Different from IndicCorpV2
        "load_kwargs": {
            "path": "oscar-corpus/OSCAR-2301",
            "name": "te",  # Telugu language code
            "streaming": True,
            "split": "train",
        },
        "max_documents": MAX_DOCUMENTS,
    }
    
    print(" OSCAR-2301 Processing Configuration:")

    
    for key, value in config.items():
        if key != "load_kwargs":
            print(f"   {key}: {value}")
    
    print("\n HuggingFace Dataset Parameters:")
    for key, value in config["load_kwargs"].items():
        print(f"   {key}: {value}")
    
    print("\nStatus: Pending HuggingFace dataset access approval")
    print("   1. Visit: https://huggingface.co/datasets/oscar-corpus/OSCAR-2301")
    print("   2. Request access to the gated dataset")
    print("   3. Set HF_TOKEN environment variable")
    print("   4. Run: python run_oscar.py")

show_task2_config()

### **Final Results Summary:**

| Metric | Achievement |
|--------|-------------|
| **Documents Processed** | 5.5 Million |
| **Sentences Tokenized** | 20,181,498 (2+ crore) |
| **Words Processed** | 249,508,148 (25+ crore) |
| **Unique Vocabulary** | 3,737,704 (37+ lakh) |
| **Storage Efficiency** | 1.66 GB compressed |
| **Processing Rate** | ~131K documents/minute |

